
## <p style='text-align: center;'> Dense matter and a hands-on section on the MUSES cyberinfrastructure </p>
### <p style='text-align: left;'> **CSQCD 2026** </p>

---

MUSES stands for Modular Unified Solver of the Equation of State. It is a cyberinfrastructure that provides tools for computing the Equation of State in nuclear astrophysics and heavy-ion collisions, as well as for computing neutron star observables.


Our main service at the moment is the [Calculation Engine (CE)](https://ce.musesframework.io/), an open-source platform that hosts several version-controlled modules for computing the EoS and neutron star properties. The CE is a web-based interface that allows users to run modules and workflows without needing to install anything on their local machines. 

The  *modules* are configurable through YAML-based input files. Several modules can be connected and run together in a *Workflow*, which is also set up in YAML. 

The CE also allows users to upload files and use them as input for modules, as well as to use files generated in previous jobs.

In this session, we will learn how to:
1. Set up workflows
2. Upload files and use them.
3. Use files from previous jobs (individual workflow executions)
---

***Let's get started:***

1. Go to the [Calculation Engine website](https://ce.musesframework.io/) .
2. Log in with your ORCID or University email.
3. Click **Request Access**. You will be redirected to the [forum](https://forum.musesframework.io/).
4. Sign up **with the same log in method as in the CE** and request access.
5. You'll be approved in a moment.
6. Refresh the CE page for access to be granted.

**If you are already done with this, [pick a module in the documentation and browse through it](https://ce.musesframework.io/docs/user/quickstart.html)**



Obs: The modules in the CE follow certain guidelines. If you have a software that you think fits well in our cyberinfrastructure, contact us in [the forum](https://forum.musesframework.io/) and take a look at the [template module](https://gitlab.com/nsf-muses/template-module/template-module).

---


#### **This session**

##### First part: Run a simple workflow [CMF $\rightarrow$  Lepton $\rightarrow$ QLIMR]

##### Second part: Run a different simple workflow [Crust DFT $\rightarrow$  Lepton $\rightarrow$ QLIMR]

#### Third part: Merge different EOSs [Crust DFT + CMF $\rightarrow$  Synthesis $\rightarrow$ QLIMR]

##### **Exercises**
#### - **Factorize the workflow and run different CMF potentials / particle populations**
#### - **Compare different EOS matching strategies**

---

**Author**: Mateus R. Pelicer

**Contact**: mateusreinke@hotmail.com

In [ ]:
import os # for environment variables
import pandas as pd # for data manipulation
import yaml # for configuration management
import time
import sys # for installing requirements
import subprocess # for cloning the repository
import getpass # for user information
import matplotlib.pyplot as plt # for plotting
from pathlib import Path # for path manipulations


In [ ]:
REPO_URL = "https://github.com/mrpelicer/muses-csqcd2026.git"
REPO_DIR = Path("/repo/muses-csqcd2026")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab.")

    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True
        )

    os.chdir(REPO_DIR)

    if Path("requirements.txt").exists():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
        )
else:
    print("Running locally.")

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

OUTPUT_PATH = repo_root / "output"
PLOTS_PATH = repo_root / "plots"
Path(OUTPUT_PATH).mkdir(exist_ok=True)
Path(PLOTS_PATH).mkdir(exist_ok=True)

print("Working directory:", Path.cwd())
print("Python path includes:", repo_root)
from calculation_engine_api import CalculationEngineApi # API client to interact with the calculation engine

**Paste your API token after getpass requests it**

In [ ]:
# Setup the environment variables
os.environ['CE_API_URL_PROTOCOL'] = 'https'
os.environ['CE_API_URL_AUTHORITY'] = 'ce.musesframework.io'
if not 'CE_API_TOKEN' in os.environ:
    os.environ['CE_API_TOKEN'] = getpass.getpass()

In [ ]:
# Set up the api object to interact with the calculation engine
api = CalculationEngineApi()

try:
    modules = api.list_modules()
    print(f"Connection ok. Found {len(modules)} modules:")
    for module in modules:
        print(f"  Module: {module['name']:<30}  url: {module['source']['url']}")
except Exception as err:
    raise RuntimeError(
        "Could not connect to the MUSES Calculation Engine. "
        "Check your token and internet connection."
    ) from err


In [ ]:
# helper functions 
def wait_for_job(api, uuid, sleep=10, timeout=6000):
    start = time.time()
    while True:
        job = api.job_list(uuid)
        if job['status'] in ['SUCCESS', 'FAILURE']:
            return job
        if time.time() - start > timeout:
            raise TimeoutError(
                f"Job {uuid} did not finish within {timeout} seconds. "
                f"Last status: {job['status']}"
            )


        print(f"Job {uuid} is still running. Status: {job['status']}")
        time.sleep(sleep)

def print_job_files(job_status):
    print(f"Job {job_status['uuid']} files:")
    for file in job_status['files']:
        print(f"  - {file['path']} (size: {file['size']} bytes)")

def download_job_outputs(api, job_status, base_output_dir):
    output_dir = os.path.join(base_output_dir, job_status['uuid'])
    os.makedirs(output_dir, exist_ok=True)
    for file in job_status['files']:
        if file['size']>0:
          print(f"File: {file['path']}")
          api.download_job_file(job_status['uuid'], path=file['path'], root_dir=base_output_dir)

--- 
---

#### **1. CMF $\rightarrow$  Lepton $\rightarrow$ QLIMR workflow**

In this first workflow, CMF produces a dense-matter equation of state. The module can be computed with nucleons, hyperons, decuplet baryons and quarks.

The Lepton module then adds electrons and muons and imposes beta equilibrium.

QLIMR then solves the stellar structure equations and returns neutron-star observables.

The workflow is set up in the YAML file below. 
- The processes define the modules that are run. Each process must have a `name`, `module` and `config`. 
- If the module takes input files, then `pipes` and/or `inputs` are also used.

You can find more information [MUSES documentation](https://ce.musesframework.io/docs/user/quickstart.html) or in [arXiv:2409.06837](https://arxiv.org/abs/2409.06837) for the CMF EOS.

In [ ]:
### define the workflow
cmf_wf_config = yaml.safe_load('''
processes:
- name: cmf
  module: cmf_solver
  config:
    computational_parameters:
      options:
        vector_potential: 4
        use_octet: true
        use_hyperons: true
        use_decuplet: false
        use_quarks: false
      variables:
        chemical_optical_potentials:
          muB_begin: 950.0
          muB_end: 1800.
          muB_step: 20.0
          muQ_begin: -280.0
          muQ_end: 0.
          muQ_step: 5.0
      output_files:
        output_Lepton: true
        output_particle_properties: true
- name: lepton-cmf
  module: lepton
  config:
    global:
      use_beta_equilibrium: true
    output:
      output_compose: true
      output_particle_properties: true
    particles:
      use_electron: true
      use_muon: true
  pipes:
    input_eos:
      label: CMF_for_Lepton_baryons_only
      module: cmf_solver
      process: cmf
    input_particle_properties:
      label: CMF_particle_properties_baryons_only
      module: cmf_solver
      process: cmf                     
- name: qlimr-cmf
  module: qlimr
  pipes:
    eos:
      label: eos_beta_equilibrium
      module: lepton
      process: lepton-cmf
  config:
    inputs:
      R_start: 0.0004
      eos_name: eos
      initial_epsilon: 150.0
      final_epsilon: 6000.0
      resolution_in_NS_M: 0.1
      resolution_in_NS_R: 0.01
    options:
      eps_sequence: true
      output_format: csv
      stable_branch: true
    outputs:
      compute_inertia: true
      compute_love: true
      compute_mass_and_radius_correction: false
      compute_quadrupole: true
      local_functions: false
components:
- type: chain
  name: cmf_workflow
  sequence:
    - cmf
    - lepton-cmf
    - qlimr-cmf
''')

In [ ]:
## Send an API request to the Calculation engine to create the workflow job
cmf_response = api.job_create(
    name=f'CMF-Lepton-QLIMR workflow',
    description='Test workflow for CMF with Lepton and QLIMR',
    config={
    "workflow": {
        "config": cmf_wf_config
    }
})
print(f'Job created with UUID: {cmf_response["uuid"]}')

In [ ]:
## Check the status of the CMF job until it is completed
cmf_status = wait_for_job(api, cmf_response['uuid'], sleep=10)
print(f'Final job status: {cmf_status["status"]}')

In [ ]:
## Download the output files from the job into ./output/
print(cmf_status)
download_job_outputs(api, cmf_status, base_output_dir=OUTPUT_PATH)


Notice that the CMF-only EOS is appropriate for dense matter but lacks a realistic crust.

This is why the pressure-energy curve does not cover the low-density region needed for a complete neutron-star EOS.

In [ ]:
# Define the data structure for the output files
eos_data = ['T', 'muB', 'muS', 'muQ', 'nB', 'nS', 'nQ', 'e', 'p', 's']
ns_data = ['Ec', 'R', 'M', 'Ibar', 'Lbar', 'Qbar', 'es_Omega', 'dReq_Omega2', 'dR_Omega2', 'dM_Omega2']

# Read the data files
cmf_eos_df = pd.read_csv(os.path.join(OUTPUT_PATH, cmf_response['uuid'], 'lepton-cmf', 'opt', 'output', 'beta_equilibrium_eos.csv'), 
                     names=eos_data, na_values=['NA', 'NAN', 'NaN', 'nan', '-NAN'], sep=',')

cmf_ns_df = pd.read_csv(os.path.join(OUTPUT_PATH, cmf_response['uuid'], 'qlimr-cmf', 'opt', 'output', 'observables.csv'),
                      names=ns_data, na_values=['NA', 'NAN', 'NaN', 'nan', '-NAN'], sep=',')


plt.plot(cmf_eos_df['e'], cmf_eos_df['p'], label='CMF')
plt.ylim(0,)
plt.xlabel('Energy Density (MeV/fm^3)')
plt.ylabel('Pressure (MeV/fm^3)')
plt.legend()
plt.savefig(PLOTS_PATH / 'cmf_eos.png')
plt.show()

plt.plot(cmf_ns_df['R'], cmf_ns_df['M'], label='CMF')
plt.xlabel('Radius (km)')
plt.ylabel('Mass (M_sun)')
plt.ylim(0.2,)
plt.legend()
plt.savefig(PLOTS_PATH / 'cmf_mr.png')
plt.show()

--- 
---

#### **2. Crust-DFT $\rightarrow$  Lepton $\rightarrow$ QLIMR workflow**

In the second workflow, crust-DFT produces an equation of state that covers both the crust and core. The module can be computed with different EOSs (different saturation properties), but it only contains nucleons as degrees of freedom.

In [ ]:
## We'll interpolate an existing hdf5 file to get the Crust-DFT EOS in MUSES format
## You can do the entire calculation by setting generate_table: true in the config, but we do not have time for it.
# We have a list of precomputed Crust-DFT EOS files in HDF5 available in the MUSES framework.

crust_dft_eos = 'fid_3_5_22'
crust_dft_input={}

if crust_dft_eos == 'fid_3_5_22':
    crust_dft_input['uuid'] = 'd1ed1c63-6192-4ac9-9cb1-a7d82cc27b72'
    crust_dft_input['checksum']= '164575f9d84c3ac087780e0219ee2e8a'                            
elif crust_dft_eos == 'fid_414_2_6_22':
    crust_dft_input['uuid'] = '9db6a7a6-746d-4d13-ba96-15522798bb25'
    crust_dft_input['checksum']= '21cdff54b747ab6d24299e13c79b424e'
elif crust_dft_eos == 'fid_450_2_6_22':
    crust_dft_input['uuid'] = 'a4f8be0f-57b6-4fe4-b460-3ef5720ead7b'
    crust_dft_input['checksum']= 'e39eb584e5587b74a90c74f9648f8e5f'
elif crust_dft_eos == 'large_mmax':
    crust_dft_input['uuid'] = '8acd4e96-e2ce-498a-a811-676c5276d915'
    crust_dft_input['checksum']= 'ce327473bdf9ae10e02eaf86e9c34f9f'
elif crust_dft_eos == 'large_sl_2_6_22':
    crust_dft_input['uuid'] = 'a4354153-84f2-4812-9c80-f501e7b057c3'
    crust_dft_input['checksum']= '0a4fcf8cc1e0a8e37eee5ca47c73c0c9'
elif crust_dft_eos == 'small_sl_2_6_22':
    crust_dft_input['uuid'] = '8e5f694e-6e96-4be0-8876-2c38f69524fd'
    crust_dft_input['checksum']= '42a33681ad010b3804eb5d3ab2f68154'
elif crust_dft_eos == 'large_r_2_6_22':
    crust_dft_input['uuid'] = 'a5f07b17-41a8-491a-a785-a2afd08ad4fb'
    crust_dft_input['checksum']= '19688cbba5f359efb2aa2f6673914285'
elif crust_dft_eos == 'small_r_2_6_22':
    crust_dft_input['uuid'] = '31d86bc1-5677-4c5e-a4e9-e088aa350092'
    crust_dft_input['checksum']= 'c47d687394f935dcb0e18c5f0e8a75ed'
elif crust_dft_eos == 'smaller_r_2_6_22':
    crust_dft_input['uuid'] = '30258dc4-d142-47e0-895c-e959d907dab7'
    crust_dft_input['checksum']= '6d6444c2ac7fc88656d41ae846aa3fe4'
print(crust_dft_input)

In [ ]:
dft_wf_config=yaml.safe_load(f'''
processes:
- name: crust_dft_eos
  module: crust_dft
  inputs:
    EOS_table:
      type: upload
      uuid: {crust_dft_input['uuid']}
      checksum: {crust_dft_input['checksum']}
  config:
    output_format: CSV
    generate_table: false 
    ext_guess: false                           
    set:
      Ye_grid_spec: 70,0.01*(i+1)
      nB_grid_spec: 301,10^(i*0.04-12)*2.0
      verbose: 0
- name: lepton-crust_dft
  module: lepton
  config:
    global:
      use_beta_equilibrium: true
    particles:
      use_electron: true
      use_muon: true
  pipes:
    input_eos:
      label: crust_dft_output
      module: crust_dft
      process: crust_dft_eos
- name: qlimr-crust_dft
  module: qlimr
  pipes:
    eos:
      label: eos_beta_equilibrium
      module: lepton
      process: lepton-crust_dft
  config:
    inputs:
      R_start: 0.0004
      eos_name: eos
      initial_epsilon: 150.0
      final_epsilon: 6000.0
      resolution_in_NS_M: 0.2
      resolution_in_NS_R: 0.02
    options:
      eps_sequence: true
      output_format: csv
      stable_branch: true
    outputs:
      compute_inertia: true
      compute_love: true
      compute_mass_and_radius_correction: false
      compute_quadrupole: true
      local_functions: false
components:
- type: chain
  name: crust_dft_beta
  sequence:
    - crust_dft_eos
    - lepton-crust_dft
    - qlimr-crust_dft
''')

In [ ]:
# Create the Crust-DFT workflow job
dft_response = api.job_create(
    name=f'Crust-DFT - Lepton - QLIMR workflow',
    description='Test workflow for CMF with lepton and QLIMR',
    config={
    "workflow": {
        "config": dft_wf_config
    }
})
print(f'Job created with UUID: {dft_response}')

In [ ]:
# Check the status of the Crust-DFT job until it is completed
dft_status = wait_for_job(api, dft_response['uuid'], sleep=10)
print(f'Final job status: {dft_status["status"]}')

In [ ]:
## Read the files
## We can read the files using the API client to get the data in a pandas dataframe by downloading it to a tmp path and reading it with pandas, without having to download the files manually
## You can get the path of the files from the response, like we showed in the previous workflow 
dft_eos_df = api.read_job_file_to_dataframe(dft_response['uuid'], 
                               path=os.path.join('lepton-crust_dft', 'opt', 'output', 'beta_equilibrium_eos.csv'),
                               names=eos_data, 
                               na_values=['NA', 'NAN', 'NaN', 'nan', '-NAN'], 
                               sep=',')

dft_ns_df = api.read_job_file_to_dataframe(dft_response['uuid'], 
                                path=os.path.join('qlimr-crust_dft', 'opt', 'output', 'observables.csv'),           
                                names=ns_data, 
                                na_values=['NA', 'NAN', 'NaN', 'nan', '-NAN'], 
                                sep=',')

Crust-DFT produces both a crust and a core EoS. The module can be computed with different nuclear functionals.

Now the pressure-energy curve covers the low-density region, but the high-density region may not be appropriate for neutron stars, since the module does not include the relevant degrees of freedom for dense matter.

In [ ]:
plt.plot(cmf_eos_df['e'], cmf_eos_df['p'], label='CMF')
plt.plot(dft_eos_df['e'], dft_eos_df['p'], label='Crust-DFT')
plt.ylim(0.,1500)
plt.legend()
plt.xlim(0.,2000)
plt.xlabel('Energy Density (MeV/fm^3)')
plt.ylabel('Pressure (MeV/fm^3)')
plt.savefig(PLOTS_PATH / 'cmf_crust_dft_eos.png')
plt.show()



plt.plot(cmf_eos_df['e'], cmf_eos_df['p'], label='CMF')
plt.plot(dft_eos_df['e'], dft_eos_df['p'], label='Crust-DFT')
plt.xscale('log')
plt.yscale('log')
plt.ylim(1e-13,1500)
plt.legend()
plt.xlim(1e-8,2000)
plt.xlabel('Energy Density (MeV/fm^3)')
plt.ylabel('Pressure (MeV/fm^3)')
plt.savefig(PLOTS_PATH / 'cmf_crust_dft_log_eos.png')
plt.show()


plt.plot(cmf_ns_df['R'], cmf_ns_df['M'], label='CMF')
plt.plot(dft_ns_df['R'], dft_ns_df['M'], label='Crust-DFT')
plt.ylim(0.5,)
plt.xlabel('Radius (km)')
plt.ylabel('Mass (M_sun)')
plt.legend()
plt.savefig(PLOTS_PATH / 'cmf_crust_dft_log_mr.png')
plt.show()

--- 
---

#### **3. Crust-DFT + CMF $\rightarrow$ Synthesis $\rightarrow$ QLIMR workflow**


CMF lacks a crust, but we can use the low density-part of Crust-DFT and the high-density part of CMF to make a full EOS and use it to compute neutron star properties.

We will learn how to: 
- Upload an EOS and use it in a workflow by uploading the CMF EOS we downloaded in the first step.
- Use existing jobs in new workflows by using the Crust-DFT EOS from the job in the second step.
- Put two EOS togetherby usingthe synthesis module to attach the crust DFT EOS to the CMF EOS

Then we will run the `QLIMR` module to compute the neutron star properties



In [ ]:
uploads = api.upload_list()
cmf_upload_path = '/school/example/cmf_eos.csv'
for upload in uploads:
    if upload['path'] == cmf_upload_path:
        api.delete_uploaded_file(upload['uuid'])
       
cmf_input = api.upload_file(file_path=os.path.join(OUTPUT_PATH, cmf_response['uuid'], 'lepton-cmf', 'opt', 'output', 'beta_equilibrium_eos.csv'), 
                            upload_path=cmf_upload_path,
                            description='CMF EOS',)
if not cmf_input:
    print("Failed to upload the CMF EOS file. You probably already have an EOS in the same path")
else:
  print(cmf_input)

The synthesis module takes two EOSs as input and allows the user to choose how to merge them (we used synthesis for the name due to the already widespread use of merging for other things, e.g. BNS merger)

We will use the simplest way to put the crust and core together: choose a density at which the crust and core are merged, and allow for a small gap in other thermodynamic quantities to make the transition smoother. Then the QLIMR interpolates the EOS and computes the neutron star properties, effectivelly filling out the pressure and energy density.

See the [Synthesis module documentation](https://ce.musesframework.io/docs/modules/synthesis/index.html) for more details on other matching procedures.

In [ ]:
match_wf_config = yaml.safe_load(f'''
processes:
- name: synthesis
  module: synthesis
  config:
    global:
      synthesis_type: attach
    attach_method:
      attach_variable: baryon_density
      attach_value: 0.05
  inputs:
    model1_BetaEq_eos:
      type: job
      uuid: {dft_response['uuid']}
      path: /lepton-crust_dft/opt/output/beta_equilibrium_eos.csv
    model2_BetaEq_eos:
      type: upload
      uuid: {cmf_input['uuid']}
      checksum: {cmf_input['checksum']}
- name: qlimr
  module: qlimr
  pipes:
    eos:
      label: eos
      module: synthesis
      process: synthesis
  config:
    inputs:
      R_start: 0.0004
      eos_name: eos
      final_epsilon: 6000.
      initial_epsilon: 150.
      resolution_in_NS_M: 0.2
      resolution_in_NS_R: 0.02
    options:
      eps_sequence: true
      output_format: csv
      stable_branch: true
    outputs:
      compute_inertia: true
      compute_love: true
      compute_mass_and_radius_correction: false
      compute_quadrupole: true
      local_functions: false
components:
- type: chain
  name: synthesis_qlimr_workflow
  sequence:
    - synthesis
    - qlimr
''')

In [ ]:
# Create the job
synthesis_response = api.job_create(
    name=f'Crust-DFT + CMF - Synthesis - QLIMR workflow',
    description='Test workflow for Synthesis with Qlimr',
    config={
    "workflow": {
        "config": match_wf_config
    }
})
print(f'Job created with UUID: {synthesis_response["uuid"]}')

In [ ]:
# Read jobs status
synthesis_status = wait_for_job(api, synthesis_response['uuid'], sleep=5)
print(f'Final job status: {synthesis_status["status"]}')

In [ ]:
# Read the output files directly from the api
eos_df = api.read_job_file_to_dataframe(synthesis_response['uuid'],
                                path=os.path.join('synthesis', 'opt', 'output', 'eos.csv'),
                                names=eos_data, 
                                na_values=['NA', 'NAN', 'NaN', 'nan', '-NAN'], 
                                sep=',')

ns_df = api.read_job_file_to_dataframe(synthesis_response['uuid'],
                                path=os.path.join('qlimr', 'opt', 'output', 'observables.csv'),
                                names=ns_data, 
                                na_values=['NA', 'NAN', 'NaN', 'nan', '-NAN'], 
                                sep=',')



In [ ]:
# Plots

plt.plot(cmf_eos_df['e'], cmf_eos_df['p'], label='CMF', lw=3)
plt.plot(dft_eos_df['e'], dft_eos_df['p'], label='Crust-DFT', lw=3)
plt.plot(eos_df['e'], eos_df['p'], label='Crust-DFT+CMF', lw=1)
plt.xscale('log')
plt.yscale('log')
plt.ylim(1e-13,1500)
plt.xlim(1e-8,2000)
plt.xlabel('Energy Density (MeV/fm^3)')
plt.ylabel('Pressure (MeV/fm^3)')
plt.legend()
plt.savefig(PLOTS_PATH / 'merged_cmf_crust_dft_log_eos.png')
plt.show()

plt.plot(cmf_ns_df['R'], cmf_ns_df['M'], label='CMF')
plt.plot(dft_ns_df['R'], dft_ns_df['M'], label='Crust-DFT')
plt.plot(ns_df['R'], ns_df['M'], label='Crust-DFT+CMF')
plt.xlabel('Radius (km)')
plt.ylabel('Mass (M_sun)')
plt.ylim(0.5,)
plt.legend()
plt.savefig(PLOTS_PATH / 'merged_cmf_crust_dft_log_mr.png')
plt.show()

### Exercise 1: Canonical neutron-star observables

For each EOS, compute:

- $M_{\max}$
- $R_{1.4}$
- $\Lambda_{1.4}$
- $\bar{I}_{1.4}$


Which quantity changes the most when adding a crust?
Which quantity changes the most when changing the high-density EOS?


### Exercise 2: From tidal deformability to Love number

QLIMR outputs $\bar{\lambda}$, also commonly written as $\Lambda$.
Use the compactness

$C = GM/(Rc^2)$

to estimate the quadrupolar Love number $k_2$:

$k_2 = \frac{3}{2}\Lambda C^5.$

Plot both $\Lambda(M)$ and $k_2(M)$.

Which one varies more rapidly with mass?
Why is $\Lambda$ so sensitive to the stellar radius?



### Exercise 3: Hyperon puzzle:

Use the CMF model with different vector potentials with and without hyperons.

You can factorize the YAML workflow file to make it easier to run different CMF configurations with the same crust EOS. 

What happens to the maximum mass when you add hyperons? What happens when you change the vector potential?

The hyperon puzzle is the fact that hyperons are expected to appear in neutron stars, but their presence tends to soften the EOS and reduce the maximum mass of neutron stars, which can be in tension with observations of massive neutron stars.
